# 02 — Baseline models

This notebook establishes simple, reproducible benchmarks. It does not tune hyperparameters. Exact duplicates are removed before splitting, and the 20% stratified test set is only used for this final baseline evaluation. Accuracy is reported, but it is misleading when 99.8% of observations are legitimate; PR-AUC (average precision) is the primary metric.

In [7]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from src.data_utils import split_features_target


## Prepare data

The split is stratified so the rare fraud class has nearly the same prevalence in both sets. Scaling is used only for Logistic Regression: its regularized, gradient-based optimization is sensitive to feature magnitudes. Tree-based Random Forest and XGBoost split on feature thresholds, so scaling is unnecessary for them.

In [3]:
df = pd.read_csv("../Dataset/creditcard_cleaned.csv")
print(df.shape)
df.head(3)

(283726, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0


In [9]:
print("Training:")
print(y_train.value_counts(normalize=True))

print("\nTesting:")
print(y_test.value_counts(normalize=True))

Training:
Class
0    0.998335
1    0.001665
Name: proportion, dtype: float64

Testing:
Class
0    0.998326
1    0.001674
Name: proportion, dtype: float64


In [8]:
X, y = split_features_target(df)
print(X.shape, y.shape)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

(283726, 30) (283726,)


In [10]:
numeric_features = X_train.columns.tolist()

logistic_pipeline = Pipeline([
    ('preprocessor', ColumnTransformer([('scale_numeric', StandardScaler(), numeric_features)], remainder='drop')),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

models = {
    'Logistic Regression': logistic_pipeline,
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, eval_metric='logloss', random_state=42, n_jobs=-1),
}


In [11]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    """Train and evaluate each model"""
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    print(f'\n{name}')
    print(classification_report(y_test, predictions, digits=3, zero_division=0))
    print('Confusion matrix:\n', confusion_matrix(y_test, predictions))
    return {
        'Model': name,
        'Accuracy': (predictions == y_test).mean(),
        'Precision': precision_score(y_test, predictions, zero_division=0),
        'Recall': recall_score(y_test, predictions, zero_division=0),
        'F1-score': f1_score(y_test, predictions, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, probabilities),
        'PR-AUC / Average Precision': average_precision_score(y_test, probabilities),
    }

baseline_results = []
for name, model in models.items():
    result = evaluate_model(
        name,
        model,
        X_train,
        y_train,
        X_test,
        y_test
    )
    baseline_results.append(result)

baseline_comparison = pd.DataFrame(baseline_results).sort_values('PR-AUC / Average Precision', ascending=False)
display(baseline_comparison.style.format({column: '{:.4f}' for column in baseline_comparison.columns[1:]}))


Logistic Regression
              precision    recall  f1-score   support

           0      0.999     1.000     1.000     56651
           1      0.846     0.579     0.688        95

    accuracy                          0.999     56746
   macro avg      0.923     0.789     0.844     56746
weighted avg      0.999     0.999     0.999     56746

Confusion matrix:
 [[56641    10]
 [   40    55]]

Random Forest
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     56651
           1      0.972     0.726     0.831        95

    accuracy                          1.000     56746
   macro avg      0.986     0.863     0.916     56746
weighted avg      0.999     1.000     0.999     56746

Confusion matrix:
 [[56649     2]
 [   26    69]]

XGBoost
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     56651
           1      0.957     0.705     0.812        95

    accuracy                         

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC,PR-AUC / Average Precision
2,XGBoost,0.9995,0.9571,0.7053,0.8121,0.9730,0.8032
1,Random Forest,0.9995,0.9718,0.7263,0.8313,0.9284,0.7973
0,Logistic Regression,0.9991,0.8462,0.5789,0.6875,0.9560,0.6920


## Feature experiments

`Amount_log` and `Hour` are candidates, not assumed improvements. Compare them with stratified cross-validation on the training set only, using average precision. Do not choose a feature set from test performance.

In [4]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
feature_sets = {
    'Original': X_train,
    'Amount_log experiment': X_train.assign(Amount_log=np.log1p(X_train['Amount'])),
    'Hour experiment': X_train.assign(Hour=(X_train['Time'] // 3600).astype(int) % 24),
}
experiment_scores = {}
for label, features in feature_sets.items():
    pipeline = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(max_iter=1000, random_state=42))])
    experiment_scores[label] = cross_val_score(pipeline, features, y_train, scoring='average_precision', cv=cv, n_jobs=-1).mean()
pd.Series(experiment_scores, name='Mean CV Average Precision').sort_values(ascending=False).to_frame()

,Mean CV Average Precision
Original,0.749749
Hour experiment,0.749384
Amount_log experiment,0.747427


The original feature set achieved the highest mean cross-validated Average Precision (0.749749). Neither Hour nor Amount_log improved Logistic Regression performance in this experiment. Therefore, the original feature set was retained for subsequent modeling.